In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l6.2 RMSNorm

> 🎯 **Goal:** Swap LayerNorm for RMSNorm, a cheaper normalizer that drops the
centering step yet trains just as well.

**Why this matters.** Normalization sits in front of every attention and MLP block
and runs constantly, so making it cheaper is a real win across a full model. Modern
LLMs (LLaMA, Mistral, T5) standardized on RMSNorm precisely because it costs less
and matches LayerNorm's quality.

**The intuition.** Think of RMSNorm as LayerNorm with one step removed. LayerNorm
does two things to each vector: it *centers* it (subtract the mean so the average is
zero) and then *scales* it (divide by the spread so the size is standardized).
RMSNorm keeps only the scaling and skips the centering. It turns out the rescaling is
what stabilizes training; the mean-subtraction was mostly along for the ride.

**The mechanics.** RMSNorm divides each vector by its root-mean-square (the square
root of the average of its squared entries), then multiplies by a learned per-channel
weight. There is no mean subtraction and no learned bias. That means fewer arithmetic
operations per call and fewer *parameters* (LayerNorm learns both a scale and a bias;
RMSNorm learns only the scale). The output has RMS approximately 1 before the learned
weight reshapes it, which is exactly the well-conditioned signal the next layer
wants.

In [ ]:
from pocketlm import RMSNorm
from torch import nn

x = torch.randn(2, 4, 8) * 5 + 3
rms = RMSNorm(8)(x)
ln = nn.LayerNorm(8)(x)
print("RMSNorm output RMS ~1:", round(rms.pow(2).mean(-1).sqrt().mean().item(), 4))
print("RMSNorm params:", sum(p.numel() for p in RMSNorm(8).parameters()),
      " LayerNorm params:", sum(p.numel() for p in nn.LayerNorm(8).parameters()))

**What just happened.** The printed RMS is approximately 1, confirming RMSNorm
standardized the magnitude of each vector even though the input was shifted (mean 3)
and stretched (times 5). The parameter counts show RMSNorm with only the scale weight
(8 numbers here) versus LayerNorm with scale plus bias (16 total). The saving per
layer is small, but a model stacks dozens of norms, and the simpler operation also
runs faster.

⚠️ **Common pitfalls**
- Do not assume RMSNorm makes the *mean* zero. It does not center, so a shifted input
  stays shifted; only the scale is controlled. That is by design.
- The small epsilon inside the square root matters. Drop it and a near-zero vector
  divides by zero and produces NaNs.
- RMSNorm has no bias to remove a constant offset, so do not reach for it expecting
  the exact behavior of LayerNorm; reach for it as a leaner replacement that trains
  equivalently.

🏋️ **Try it yourself.** Change the input shift and scale (for example `* 20 + 100`)
and rerun. The output RMS should still land near 1 regardless of how extreme the
input is, which is the whole point of normalization.

In [ ]:
rms_out = RMSNorm(8)(x)
assert abs(rms_out.pow(2).mean(-1).sqrt().mean().item() - 1.0) < 0.05
assert sum(p.numel() for p in RMSNorm(8).parameters()) < \
       sum(p.numel() for p in nn.LayerNorm(8).parameters())